# PMF Baseline — H&M Sampled Dataset

Adapts the Probabilistic Matrix Factorization template to the H&M sampled data
(cached from `hm_sampled/`). Columns: `customer_id`, `product_code`, `bought`.


In [2]:
import pandas as pd
import torch
import numpy as np
import math
import os
import itertools


## 1. Load Sampled H&M Data


In [4]:
# ── Load sampled H&M dataset from cache ───────────────────────────────────────
TARGET_USERS     = 1000
MIN_INTERACTIONS = 20
N_QUANTILES      = 4
SAMPLED_DATA_DIR = r"/Users/haowen/Documents/Decentral RS/rebuttal/code/hm_sampled"

cache_tag  = f"u{TARGET_USERS}_m{MIN_INTERACTIONS}_s42"
train_path = os.path.join(SAMPLED_DATA_DIR, f"train_{cache_tag}.csv")
val_path   = os.path.join(SAMPLED_DATA_DIR, f"val_{cache_tag}.csv")
test_path  = os.path.join(SAMPLED_DATA_DIR, f"test_{cache_tag}.csv")
meta_path  = os.path.join(SAMPLED_DATA_DIR, f"meta_{cache_tag}.csv")

assert all(os.path.exists(p) for p in [train_path, val_path, test_path, meta_path]), (
    f"Cache not found in '{SAMPLED_DATA_DIR}/'. Run the sampling notebook first."
)

train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)
meta_df  = pd.read_csv(meta_path)

n_users = int(meta_df.loc[meta_df["key"] == "n_users", "value"].iloc[0])
n_items = int(meta_df.loc[meta_df["key"] == "n_items", "value"].iloc[0])

print(f"Loaded : {cache_tag}")
print(f"Users  : {n_users} | Items : {n_items}")
print(f"Train  : {len(train_df)} | Val : {len(val_df)} | Test : {len(test_df)}")
train_df.head()


Loaded : u1000_m20_s42
Users  : 948 | Items : 7418
Train  : 38708 | Val : 9677 | Test : 15696


,customer_id,product_code,bought
0,115,339,4
1,634,546,1
2,630,230,2
3,233,222,2
4,778,9,1


## 2. Build Rating Matrices

Missing entries are set to `-1` and masked out in the loss.


In [6]:
# ── Scale to [0,1] based on train min/max ─────────────────────────────────────
min_rating = float(train_df["bought"].min())
max_rating = float(train_df["bought"].max())

def make_matrix(df, n_users, n_items, min_r, max_r):
    """Dense n_users x n_items tensor. Observed entries scaled to [0,1]; missing = -1."""
    mat   = torch.full((n_users, n_items), -1.0)
    users = torch.tensor(df["customer_id"].values,  dtype=torch.long)
    items = torch.tensor(df["product_code"].values, dtype=torch.long)
    vals  = torch.tensor(
        (df["bought"].values - min_r) / (max_r - min_r), dtype=torch.float32
    )
    mat[users, items] = vals
    return mat

train_matrix = make_matrix(train_df, n_users, n_items, min_rating, max_rating)
val_matrix   = make_matrix(val_df,   n_users, n_items, min_rating, max_rating)
test_matrix  = make_matrix(test_df,  n_users, n_items, min_rating, max_rating)

print(f"Train matrix : {n_users} x {n_items}, "
      f"{(train_matrix != -1).sum().item()} observed")
print(f"Val   matrix : {(val_matrix   != -1).sum().item()} observed")
print(f"Test  matrix : {(test_matrix  != -1).sum().item()} observed")


Train matrix : 948 x 7418, 38708 observed
Val   matrix : 9677 observed
Test  matrix : 15696 observed


## 3. PMF Model & Loss


In [8]:
class PMF(torch.nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, u_features, v_features):
        """Returns sigmoid(U V^T), shape (n_users, n_items)."""
        return torch.sigmoid(torch.mm(u_features, v_features.t()))


class PMFLoss(torch.nn.Module):
    def __init__(self, lam_u=0.1, lam_v=0.1):
        super().__init__()
        self.lam_u = lam_u
        self.lam_v = lam_v

    def forward(self, matrix, predicted, u_features, v_features):
        mask             = (matrix != -1).float()
        prediction_error = torch.sum(((matrix - predicted) ** 2) * mask)
        u_reg            = self.lam_u * torch.sum(u_features.norm(dim=1))
        v_reg            = self.lam_v * torch.sum(v_features.norm(dim=1))
        return prediction_error + u_reg + v_reg


def compute_rmse(matrix, predicted, min_r, max_r):
    """RMSE on observed entries, rescaled to original purchase-count units."""
    mask = (matrix != -1).float()
    n    = mask.sum()
    if n == 0:
        return float("nan")
    pred_sc = predicted * (max_r - min_r) + min_r
    true_sc = matrix   * (max_r - min_r) + min_r
    return math.sqrt((((pred_sc - true_sc) ** 2 * mask).sum() / n).item())


print("PMF, PMFLoss, compute_rmse defined.")


PMF, PMFLoss, compute_rmse defined.


## 4. Parameter Tuning -- Grid Search

Exhaustively trains every combination in `LR_GRID x LAM_GRID x LATENT_GRID`
and selects the combo with the lowest best validation RMSE.
Best values are written into `lr`, `lam`, `latent_vectors` automatically.

Run **once** after building the matrices (cells 3-5), then run the training cell.


In [10]:
# ── Grid ─────────────────────────────────────────────────────────────────────
LR_GRID     = [0.001, 0.01,  0.05]
LAM_GRID    = [0.01,  0.1,   0.3 ]   # shared lam_u = lam_v
LATENT_GRID = [10,    20,    40  ]   # latent dimension K
TUNE_EPOCHS = 50
TUNE_PAT    = 5                      # early-stop patience

param_grid = list(itertools.product(LR_GRID, LAM_GRID, LATENT_GRID))
print(f"Grid search: {len(param_grid)} combinations x up to {TUNE_EPOCHS} epochs")
print(f"{'#':>3}  {'lr':>7}  {'lam':>6}  {'K':>4}  {'best val RMSE':>14}")
print("-" * 42)

grid_results = []

for _k, (_lr, _lam, _K) in enumerate(param_grid, 1):
    torch.manual_seed(0)

    _uf = torch.randn(n_users, _K, requires_grad=True)
    _vf = torch.randn(n_items, _K, requires_grad=True)
    _uf.data.mul_(0.01)
    _vf.data.mul_(0.01)

    _model     = PMF()
    _loss_fn   = PMFLoss(lam_u=_lam, lam_v=_lam)
    _optimizer = torch.optim.Adam([_uf, _vf], lr=_lr)

    _best_val, _wait = float("inf"), 0

    for _ep in range(TUNE_EPOCHS):
        _optimizer.zero_grad()
        _pred = _model(_uf, _vf)
        _loss = _loss_fn(train_matrix, _pred, _uf, _vf)
        _loss.backward()
        _optimizer.step()

        with torch.no_grad():
            _pred_eval = _model(_uf, _vf)
            _val_rmse  = compute_rmse(val_matrix, _pred_eval, min_rating, max_rating)

        if _val_rmse < _best_val:
            _best_val, _wait = _val_rmse, 0
        else:
            _wait += 1
            if _wait >= TUNE_PAT:
                break

    grid_results.append((_best_val, _lr, _lam, _K))
    _marker = " <--" if _best_val == min(r[0] for r in grid_results) else ""
    print(f"{_k:>3}  {_lr:>7.4f}  {_lam:>6.3f}  {_K:>4d}  "
          f"{_best_val:>14.4f}{_marker}")

# ── Best combo ────────────────────────────────────────────────────────────────
best_val_t, best_lr, best_lam, best_K = min(grid_results, key=lambda x: x[0])

print(f"\nBest val RMSE  : {best_val_t:.4f}")
print(f"  lr             = {best_lr}")
print(f"  lam            = {best_lam}")
print(f"  latent_vectors = {best_K}")

lr             = best_lr
lam            = best_lam
latent_vectors = best_K
print("\nHyperparameters updated. Run the training cell below.")


Grid search: 27 combinations x up to 50 epochs
  #       lr     lam     K   best val RMSE
------------------------------------------
  1   0.0010   0.010    10          9.0738 <--
  2   0.0010   0.010    20          9.0402 <--
  3   0.0010   0.010    40          8.9527 <--
  4   0.0010   0.100    10          9.1388
  5   0.0010   0.100    20          9.1179
  6   0.0010   0.100    40          8.8862 <--
  7   0.0010   0.300    10          9.1388
  8   0.0010   0.300    20          9.1388
  9   0.0010   0.300    40          9.1388
 10   0.0100   0.010    10          3.4895 <--
 11   0.0100   0.010    20          3.0741 <--
 12   0.0100   0.010    40          2.8654 <--
 13   0.0100   0.100    10          3.5370
 14   0.0100   0.100    20          2.5469 <--
 15   0.0100   0.100    40          2.4993 <--
 16   0.0100   0.300    10          9.1388
 17   0.0100   0.300    20          9.1388
 18   0.0100   0.300    40          9.1388
 19   0.0500   0.010    10          2.8672
 20   0.0500  

## 5. Training


In [12]:
# ── Hyperparameters (set by tuning cell, or defaults below) ──────────────────
if 'latent_vectors' not in dir(): latent_vectors = 20
if 'lr'  not in dir(): lr  = 0.01
if 'lam' not in dir(): lam = 0.1
num_epoch = 100
patience  = 10

# ── Initialise parameters ─────────────────────────────────────────────────────
torch.manual_seed(0)
user_features = torch.randn(n_users, latent_vectors, requires_grad=True)
item_features = torch.randn(n_items, latent_vectors, requires_grad=True)
user_features.data.mul_(0.01)
item_features.data.mul_(0.01)

pmf       = PMF()
pmfloss   = PMFLoss(lam_u=lam, lam_v=lam)
optimizer = torch.optim.Adam([user_features, item_features], lr=lr)

# ── Training loop with early stopping ────────────────────────────────────────
best_val_rmse  = float("inf")
best_uf        = None
best_vf        = None
patience_count = 0

for epoch in range(num_epoch):
    optimizer.zero_grad()
    pred = pmf(user_features, item_features)
    loss = pmfloss(train_matrix, pred, user_features, item_features)
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        pred_eval  = pmf(user_features, item_features)
        train_rmse = compute_rmse(train_matrix, pred_eval, min_rating, max_rating)
        val_rmse   = compute_rmse(val_matrix,   pred_eval, min_rating, max_rating)

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | loss: {loss.item():.3f} "
              f"| train RMSE: {train_rmse:.4f} | val RMSE: {val_rmse:.4f}")

    if val_rmse < best_val_rmse:
        best_val_rmse  = val_rmse
        best_uf        = user_features.data.clone()
        best_vf        = item_features.data.clone()
        patience_count = 0
    else:
        patience_count += 1
        if patience_count >= patience:
            print(f"Early stopping at epoch {epoch}.")
            break

with torch.no_grad():
    user_features.data.copy_(best_uf)
    item_features.data.copy_(best_vf)

print(f"\nBest val RMSE: {best_val_rmse:.4f}")


Epoch   0 | loss: 9018.544 | train RMSE: 9.1430 | val RMSE: 9.1389
Epoch  10 | loss: 8322.877 | train RMSE: 8.5931 | val RMSE: 8.6395
Epoch  20 | loss: 3888.515 | train RMSE: 5.1690 | val RMSE: 5.5507
Epoch  30 | loss: 1313.830 | train RMSE: 1.6424 | val RMSE: 2.8094
Epoch  40 | loss: 1298.052 | train RMSE: 1.0420 | val RMSE: 2.5003
Epoch  50 | loss: 1263.584 | train RMSE: 1.1303 | val RMSE: 2.5529
Early stopping at epoch 51.

Best val RMSE: 2.4993


## 6. Test Evaluation


In [14]:
with torch.no_grad():
    pred_test = pmf(user_features, item_features)
    test_rmse = compute_rmse(test_matrix, pred_test, min_rating, max_rating)

    mask     = (test_matrix != -1).float()
    n_obs    = mask.sum()
    pred_sc  = pred_test   * (max_rating - min_rating) + min_rating
    true_sc  = test_matrix * (max_rating - min_rating) + min_rating
    test_mae = (torch.abs(pred_sc - true_sc) * mask).sum() / n_obs

print(f"Test RMSE : {test_rmse:.4f}")
print(f"Test MAE  : {test_mae.item():.4f}")


Test RMSE : 1.7451
Test MAE  : 1.0002
